This notebook is the nodes extraction note book this will be a manual approach but will be further automated

Mongo db ping

In [3]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_ARTICLES_NAME")
NODES_COLLECTION_NAME = os.getenv("MONGO_NODES_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME, NODES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles
    nodes_collection = db[NODES_COLLECTION_NAME]  # Stores extracted nodes

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas! Database: tuone


Open AI ping

In [4]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)

def ping_openai(client):
    try:
        response = client.models.list()
        print("✅ Successfully connected to OpenAI API!")
        print(f"Available Models: {[model.id for model in response.data]}")
    except Exception as e:
        print(f"❌ OpenAI API Connection Error: {e}")
ping_openai(client)

✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4.5-preview', 'omni-moderation-2024-09-26', 'gpt-4.5-preview-2025-02-27', 'gpt-4o-mini-audio-preview-2024-12-17', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4o-audio-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-4o-mini-realtime-preview', 'o1-mini-2024-09-12', 'o1-preview-2024-09-12', 'o1-mini', 'o1-preview', 'gpt-4o-mini-audio-preview', 'whisper-1', 'gpt-4-turbo', 'gpt-4o-realtime-preview-2024-10-01', 'gpt-4', 'text-embedding-3-large', 'babbage-002', 'chatgpt-4o-latest', 'tts-1-hd-1106', 'gpt-4o-audio-preview-2024-12-17', 'tts-1-hd', 'o3-mini', 'gpt-4o-mini-2024-07-18', 'gpt-4o-2024-11-20', 'tts-1', 'tts-1-1106', 'gpt-4-turbo-2024-04-09', 'o3-mini-2025-01-31', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4o-2024-08-06', 'gpt-4o-mini', 'gpt-4o-2024-05-13', 'o1', 'gpt-3.5-turbo-instruct', 'o1-2024-12-17', 'gpt-4o', 'gpt-3.5-turbo-instruct-0914', 'gpt-3.5-turbo-0125', 'gpt-4o-realtime-preview-20

Get the first five articles from mongo db

In [5]:
import json
from bson import json_util
def fetch_articles(limit=10):
    try:
        # Fetch articles with an optional limit
        articles_cursor = articles_collection.find().limit(limit)
        articles = list(articles_cursor)

        if articles:
            print(f"✅ Retrieved {len(articles)} articles from MongoDB.\n")
            for idx, article in enumerate(articles, start=1):
                print(f"--- Article {idx} ---")
                # Convert BSON to JSON using json_util
                article_json = json.dumps(article, indent=4, default=json_util.default)
                print(article_json)
        else:
            print("⚠️ No articles found in the collection.")

        return articles

    except Exception as e:
        print(f"❌ Error fetching articles: {e}")
        return []


# Fetch and print articles in JSON format
articles = fetch_articles(limit=5)

✅ Retrieved 5 articles from MongoDB.

--- Article 1 ---
{
    "_id": {
        "$oid": "67b1e41e0554959fd725edf6"
    },
    "title": "Mitsubishi Chemical expands production capacities",
    "paragraphs": [
        {
            "p1": "Tokyo-based Mitsubishi Chemical wants to boost up its battery electrolyte business. It plans several strategic steps due to growing demand on the electric vehicle market.",
            "p2": "The company plans to restarting production in a factory based in Britain and doubling its output in the States. Part of the reconstruction plan is also to close a factory in China. The company is said to be even willing to temporarily limit its mobile device battery production.",
            "p3": "Initially, the UK plant has built batteries since 2012. However, the anticipated demand did not arise, so the production lines were stopped in March 2016.asia.nikkei.com",
            "p4": "Your email address will not be published.Required fields are marked*",
          

Read prompts from file only

In [16]:
def read_prompt_from_file_only(file_path):
    with open(file_path, 'r') as file:
        prompt = file.read()
    return prompt

Combine the articles

In [17]:
def combine_paragraphs(article):
    paragraphs = article.get('paragraphs', [])
    # Handle missing or empty paragraphs
    if not paragraphs:
        print("⚠️ No paragraphs found in the article.")
        return ""

    combined_text = ""
    for para_obj in paragraphs:
        for key in sorted(para_obj.keys()):
            combined_text += para_obj[key].strip() + " "

    return combined_text.strip()


Fetch a single function

In [9]:
def fetch_single_article(skip=0):
    try:
        article = articles_collection.find().skip(skip).limit(1)
        article = list(article)

        if article:
            article_json = json.dumps(article[0], indent=4, default=json_util.default)
            print(f"✅ Retrieved Article {skip + 1}:\n{article_json}\n")
            return article[0]
        else:
            print("⚠️ No more articles found in the collection.")
            return None

    except Exception as e:
        print(f"❌ Error fetching article: {e}")
        return None

Extracting nodes from articles

In [12]:
PROMPT_PATH = "initial_node_extraction.txt"
def extract_nodes_test(text, PROMPT_PATH):

    prompt = read_prompt_from_file_only(PROMPT_PATH)

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": f'''
            Here is the article: {text}
            '''
            }
        ],
        temperature=0)

    # Ensure JSON is properly loaded
    extracted_nodes = json.loads(completion.choices[0].message.content)
    print(f"✅ Retrieved nodes {extracted_nodes}\n")
    # Extract the list of nodes correctly
    if "nodes" in extracted_nodes:
        return extracted_nodes["nodes"]
    else:
        raise ValueError("Expected 'nodes' key in LLM output but not found.")

Processing function

In [21]:
import json

def extract_nodes(article_id, text, PROMPT_PATH):
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            temperature=0
        )

        # Parse the JSON response
        extracted_data = json.loads(completion.choices[0].message.content)
        print(f"✅ Retrieved nodes {extracted_data}\n")

        if "nodes" not in extracted_data:
            raise ValueError("Expected 'nodes' key in LLM output but not found.")

        nodes = extracted_data["nodes"]

        # Format nodes for MongoDB insertion
        formatted_nodes = []
        for node in nodes:
            location_data = node.get("location", {})  # Get location or default to empty dict
            formatted_location = {
                "city": location_data.get("city", ""),  # Ensure city is always a string
                "country": location_data.get("country", "")  # Ensure country is always a string
            }

            formatted_nodes.append({
                "article_id": article_id,  # Reference to the source article
                "id": node.get("id"),  # Unique ID of the node
                "type": node.get("type"),  # Type of the entity (company, factory, etc.)
                "name": node.get("name"),  # Name of the entity
                "parent_company": node.get("parent_company"),  # Parent company if applicable
                "subsidiaries": node.get("subsidiaries", []),  # Ensure it's always a list
                "location": formatted_location,  # Fixed location with city & country
                "products": node.get("products", []),  # Ensure it's always a list
            })

        return formatted_nodes

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return []


In [26]:
def process_articles(limit, offset, PROMPT_PATH):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find({}, {"_id": 1, "paragraphs": 1})  # Fetch paragraphs instead of text
        .sort("_id", 1)  # Sort by MongoDB ObjectId (ascending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)   # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    # Ensure MongoDB collection has an index for fast lookups
    nodes_collection.create_index("article_id", unique=False)  # Allow multiple nodes per article

    for article in articles_to_process:
        articleID = str(article["_id"])  # Convert ObjectId to string

        # Extract text from paragraphs
        text = combine_paragraphs(article)

        # Skip if no text is extracted
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            # Extract and format nodes
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH)

            # Ensure extracted nodes are valid before insertion
            if not formatted_nodes:
                print(f"⚠️ No nodes extracted for Article ID: {articleID}. Skipping insertion.")
                continue

            # Assign article ID to each node if not already present
            for node in formatted_nodes:
                node["article_id"] = articleID  # Ensure each node has an article reference

            # Insert extracted nodes into MongoDB
            nodes_collection.insert_many(formatted_nodes)  # Bulk insert formatted nodes

            # Log success and number of nodes inserted
            print(f"✅ Completed Article ID: {articleID}. Inserted {len(formatted_nodes)} nodes.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")


PROCESS

In [28]:
PROMPT_PATH = "initial_node_extraction.txt"
#
process_articles(5,1, PROMPT_PATH)

🔹 Processing 5 articles (Offset: 1)
📌 Processing Article ID: 67b1e41f0554959fd725edf7
✅ Retrieved nodes {'nodes': [{'id': 'byd', 'type': 'company', 'name': 'BYD', 'parent_company': None, 'subsidiaries': []}, {'id': 'factory_shanghai', 'type': 'factory', 'name': 'BYD Battery Recycling Facility', 'location': {'city': 'Shanghai', 'country': 'China'}, 'products': ['battery recycling']}]}

✅ Completed Article ID: 67b1e41f0554959fd725edf7. Inserted 2 nodes.
📌 Processing Article ID: 67b1e41f0554959fd725edf8
✅ Retrieved nodes {'nodes': [{'id': 'basf', 'type': 'company', 'name': 'BASF', 'parent_company': None, 'subsidiaries': ['BASF Toda America']}, {'id': 'toda', 'type': 'company', 'name': 'Toda', 'parent_company': None, 'subsidiaries': []}, {'id': 'basf_toda_jv', 'type': 'joint_venture', 'name': 'BASF & Toda Joint Venture'}, {'id': 'factory_elyria', 'type': 'factory', 'name': 'BASF Toda Factory in Elyria', 'location': {'city': 'Elyria', 'country': 'USA'}, 'products': ['NCM cathode materials',